In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import random

import sys

sys.path.append("../../mapqc/")
import mapqc
import plotting

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
path_raw = "../../data/scPoli_PBMC/pbmc_raw.h5ad"
path_latent_cell = "../../data/scPoli_PBMC/pbmc_latent_cell.h5ad"
path_latent_sample = "../../data/scPoli_PBMC/pbmc_latent_sample.h5ad"

In [ ]:
adata = sc.read_h5ad(path_latent_cell)

In [ ]:
adata

In [ ]:
sc.pl.umap(
    adata,
    color=["batch_czi", "cell_type", "cell_type_for_integration", "disease"],
    wspace=0.75,
    ncols=2,
)

In [ ]:
disease_per_ds = pd.crosstab(adata.obs.disease, adata.obs.batch_czi)

In [ ]:
dataset_has_normal_and_disease = ((disease_per_ds != 0).sum(axis=0) > 1) & (
    disease_per_ds.loc["normal", :] != 0
)

In [ ]:
datasets_to_try = dataset_has_normal_and_disease.loc[
    dataset_has_normal_and_disease.values
].index.tolist()

In [ ]:
datasets_to_try

In [ ]:
disease_per_ds.loc[:, datasets_to_try]

In [ ]:
adata.obs_names_make_unique()

In [ ]:
query_dataset = "10_1016_j_cell_2021_02_018"

In [ ]:
adata.obs["reference_or_query"] = "reference"
adata.obs.loc[adata.obs.batch_czi == query_dataset, "reference_or_query"] = "query"

In [ ]:
print(adata.obs.loc[adata.obs.reference_or_query == "query", "disease"].unique())

In [ ]:
ref_q_key = "reference_or_query"
ref_cat = "reference"
query_cat = "query"
condition_key = "disease"
batch_key = "batch_czi"
sample_key = "sample_ID_czi"
emb_key = "X"
coi = "normal"

In [ ]:
adata.obs.groupby([ref_q_key, condition_key]).agg({sample_key: "nunique"}).loc[
    query_cat, :
]

In [ ]:
adata_healthy = adata[
    (adata.obs[ref_q_key] == ref_cat) | (adata.obs[condition_key] == coi), :
].copy()

In [ ]:
adata_healthy

In [ ]:
n_nhoods = 100
k = 5000

In [ ]:
query_cell_idc = np.where(adata_healthy.obs[ref_q_key] == query_cat)[0].tolist()
random.seed(0)
center_cell_idc = random.sample(query_cell_idc, k=n_nhoods)
center_cells = np.array(adata_healthy.obs.iloc[center_cell_idc, :].index)

In [ ]:
results = mapqc.mapQC_results(
    adata=adata_healthy,
    center_cells=center_cells,
    k=k,
    emb_key=emb_key,
    sample_key=sample_key,
    ref_q_key=ref_q_key,
    ref_cat=ref_cat,
    query_cat=query_cat,
    batch_key=batch_key,
    exclude_same_batch_for_ref_samples=True,
    min_n_samples_q=3,
)

In [ ]:
results.calculate_sample_dists_to_ref_samples()

In [ ]:
results.calculate_nhood_statistics()

In [ ]:
sc.pl.umap(
    results.adata,
    color=ref_q_key,
    palette={
        results.ref_cat: "tab:orange",
        results.query_cat: "tab:blue",
    },
    title=f"{query_dataset} ({coi})",
    frameon=False,
)

In [ ]:
plotting.plot_umap_filtering_info(results)

In [ ]:
plotting.plot_umap_significance(results, title=f"{query_dataset} significance")

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(results)